# Flux post-processing chain: merged IRGA72 results (2016-2026.09)

**notebook version**: `1` (26 Oct 2026)

## ℹ️ About this notebook

This notebook merges the results of the `IRGA72` flux chain into one file, on the timestamp:

| Source | What it adds |
|---|---|
| `05_FPPC_IRGA72_NEE_2016-2026.09` | `NEE`: quality flags, storage term, USTAR scenarios, gap-filled series, their uncertainty, and the partitioning into `GPP` and `RECO` |
| `25_FPPC_IRGA72_LE_2016-2026.09` | `LE`: quality flags, gap-filled series, and `ET` derived from them |
| `35_FPPC_IRGA72_H_2016-2026.09` | `H`: quality flags and gap-filled series |

Each flux notebook writes only the columns it adds, and this file holds only these columns. A
column that more than one of them computes (potential radiation, the day and night flags) is
checked to be identical and kept once. The Level-1 fluxes (`20_L1_FLUXES/IRGA72/99`) and the
meteo (`10_METEO/30_PRODUCTS/99`) are not part of this file; `80_FINALIZE` adds them, together with
this file, to the final set.

## ⚙️ Settings

In [1]:
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
FLUXDIR = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA72"
FILES = {
    'NEE': rf"{FLUXDIR}\05_FPPC_IRGA72_NEE_2016-2026.09.parquet",
    'LE': rf"{FLUXDIR}\25_FPPC_IRGA72_LE_2016-2026.09.parquet",
    'H': rf"{FLUXDIR}\35_FPPC_IRGA72_H_2016-2026.09.parquet",
}
# The period of the flux chain: from 1 January 2016, before the first IRGA72 record, to the end of
# the record.
PERIOD_START, PERIOD_END = '2016-01-01 00:15', '2026-09-30 23:45'

OUTNAME = "99_FPPC_IRGA72_MERGED_2016-2026.09"  # same name as this notebook
OUTPATH = FLUXDIR

UMOL_TO_GC = 12.011 * 1e-6 * 1800  # NEE, umol m-2 s-1 -> gC m-2 per half-hour
WM2_TO_MJ = 1800 * 1e-6  # LE and H, W m-2 -> MJ m-2 per half-hour

In [2]:
from datetime import datetime
import importlib.metadata

import diive as dv
import numpy as np
import pandas as pd

NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-10-29 21:56:29
diive version: v0.93.0


## 📂 Load data

In [3]:
frames = {k: dv.load_parquet(filepath=v) for k, v in FILES.items()}
index = pd.date_range(PERIOD_START, PERIOD_END, freq='30min', name='TIMESTAMP_MIDDLE')
for name, frame in frames.items():
    assert frame.index.equals(index), f"{name} does not run from {PERIOD_START} to {PERIOD_END} in 30 min"
    print(f"{name:>4}: {frame.shape[0]:,} records x {frame.shape[1]} columns, "
          f"{frame.index[0]} -> {frame.index[-1]}")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA72\05_FPPC_IRGA72_NEE_2016-2026.09.parquet (0.432 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA72\25_FPPC_IRGA72_LE_2016-2026.09.parquet (0.033 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA72\35_FPPC_IRGA72_H_2016-2026.09.parquet (0.027 seconds).

 NEE: 188,448 records x 121 columns, 2016-01-01 00:15:00 -> 2026-09-30 23:45:00
  LE: 188,448 records x 45 columns, 2016-01-01 00:15:00 -> 2026-09-30 23:45:00
   H: 188,448 records x 43 columns, 2016-01-01 00:15:00 -> 2026-09-30 23:45:00


## Merge

In [4]:
def add_new_columns(main, frame, name, source):
    """Add the columns of frame that main does not have yet, on main's timestamps.

    A column that main already has must hold the same values in frame, so that no result is
    dropped silently. Returns the extended frame and records each new column's source.
    """
    assert frame.index.isin(main.index).all(), f"{name} has timestamps outside the main index"
    frame = frame.reindex(main.index)
    new = [c for c in frame.columns if c not in main.columns]
    for c in frame.columns.difference(new):
        a, b = main[c], frame[c]
        both = a.notna() & b.notna()
        assert np.allclose(a[both], b[both], equal_nan=True), f"{c} differs between main and {name}"
    for c in new:
        source[c] = name
    print(f"{name}: {len(new)} new columns, {frame.shape[1] - len(new)} already present")
    return pd.concat([main, frame[new]], axis=1)

In [5]:
merged = pd.DataFrame(index=index)
SOURCE = {}
for name in ('NEE', 'LE', 'H'):
    merged = add_new_columns(merged, frames[name], name, SOURCE)
assert merged.columns.is_unique
fullset = merged  # name used by the export cell
print(f"Merged: {merged.shape[0]:,} records x {merged.shape[1]} columns")

NEE: 121 new columns, 0 already present
LE: 42 new columns, 3 already present
H: 40 new columns, 3 already present
Merged: 188,448 records x 203 columns


### 📋 Table: columns per source

In [6]:
display(pd.Series(SOURCE).value_counts().reindex(list(FILES)).rename('columns').to_frame())

,columns
NEE,121
LE,42
H,40


### 📋 Table: annual sums of the gap-filled fluxes

`NEE` uses the CUT_50 scenario; `LE`, `ET` and `H` are not USTAR-filtered (`CUT_NONE`). The last
year ends on 30 September.

In [7]:
MAIN = {
    'NEE (gC m-2)': ('NEE_L3.1_L3.3_CUT_50_QCF_gf{m}', UMOL_TO_GC),
    'LE (MJ m-2)': ('LE_L3.1_L3.3_CUT_NONE_QCF_gf{m}', WM2_TO_MJ),
    'ET (mm)': ('ET_L3.1_L3.3_CUT_NONE_QCF_gf{m}', 1),
    'H (MJ m-2)': ('H_L3.1_L3.3_CUT_NONE_QCF_gf{m}', WM2_TO_MJ),
}
METHODS = {'XGBoost': 'XG', 'MDS': 'MDS'}
_year = merged.index.year
_rows = {}
for _var, (_pattern, _gain) in MAIN.items():
    for _mname, _m in METHODS.items():
        _col = _pattern.format(m=_m)
        assert merged[_col].notna().all(), f"{_col} has gaps"
        _rows[(_var, _mname)] = (merged[_col] * _gain).groupby(_year).sum()
ANNUAL = pd.DataFrame(_rows).T
ANNUAL.index.names = ['variable', 'method']
ANNUAL.columns.name = 'year'
display(ANNUAL.round(0).astype(int))

year                  2016  2017  2018  2019  2020  2021  2022  2023  2024  \
variable     method                                                          
NEE (gC m-2) XGBoost  -119    76   -97  -282  -255   -13    -1    15   -36   
             MDS      -284    25  -197  -314  -322   -44   -61   -13   -44   
LE (MJ m-2)  XGBoost  1502  1517  1406  1464  1482  1349  1532  1581  1550   
             MDS      1588  1517  1414  1435  1467  1338  1520  1579  1545   
ET (mm)      XGBoost   611   617   572   595   602   548   623   643   630   
             MDS       645   617   575   584   596   543   618   642   628   
H (MJ m-2)   XGBoost  1116  1302  1298  1172  1381  1209  1432  1022   792   
             MDS      1070  1295  1283  1153  1380  1221  1443  1016   800   

year                  2025  2026  
variable     method               
NEE (gC m-2) XGBoost  -336  -285  
             MDS      -433  -336  
LE (MJ m-2)  XGBoost  1610  1416  
             MDS      1598  1391  
ET (mm)      XGBoost   654   577  
             MDS       650   567  
H (MJ m-2)   XGBoost  1053  1366  
             MDS      1067  1365

## 💾 Export

The merged file as parquet, and next to it an `_INFO.md` file that lists every column with its source.

In [8]:
filepath = dv.save_parquet(data=merged, filename=OUTNAME, outpath=OUTPATH)

_lines = [f"# {OUTNAME}", "",
          f"{fullset.index[0]} to {fullset.index[-1]} ({fullset.index.name}, UTC+01:00), "
          f"{fullset.shape[0]:,} records x {fullset.shape[1]} columns.",
          f"Written {datetime.now():%Y-%m-%d %H:%M} by `{OUTNAME}.ipynb`.", "",
          "| source | file |", "|---|---|"]
_lines += [f"| {k} | `{v}` |" for k, v in FILES.items()]
for _name in FILES:
    _cols = [c for c, s in SOURCE.items() if s == _name]
    _lines += ["", f"## {_name} ({len(_cols)} columns)", ""] + [f"- `{c}`" for c in _cols]
with open(rf"{OUTPATH}\{OUTNAME}_INFO.md", "w", encoding="utf-8") as _f:
    _f.write("\n".join(_lines) + "\n")
print(f"Wrote {OUTPATH}\\{OUTNAME}_INFO.md")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA72\99_FPPC_IRGA72_MERGED_2016-2026.09.parquet (1.662 seconds).

Wrote F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROCESSING_CHAIN\IRGA72\99_FPPC_IRGA72_MERGED_2016-2026.09_INFO.md


***

# End of notebook.

In [9]:
NOTEBOOK_END = datetime.now()
print(f"Runtime: {NOTEBOOK_END - NOTEBOOK_START}")

Runtime: 0:00:03.092942
